# Demo 2 - Demonstração - Aprendizagem Supervisionada

### MBA em AI Engineering & Multi Agents - Machine Learning Foundations & Classic Models.

### Algoritmo para classificação de transações em possíveis fraudes/não fraude.

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. - https://www.kaggle.com/pwnpen/payment

Este notebook serve como uma demonstração prática dos conceitos de Aprendizagem Supervisionada, focando na construção e avaliação de modelos de classificação para detecção de fraudes em transações de pagamento. Através deste exemplo, exploraremos as etapas desde o pré-processamento dos dados até a aplicação de algoritmos como Regressão Logística e Árvore de Decisão, e a interpretação de métricas de desempenho.

**Objetivo:** Classificar transações como fraudulentas (1) ou não fraudulentas (0) utilizando diferentes modelos de Machine Learning.

**Dados:** Utilizaremos um dataset de transações de pagamento contendo informações como idade da conta, número de itens, tempo local, método de pagamento e idade do método de pagamento. A coluna 'label' indica se a transação é fraude (1) ou não (0).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score, confusion_matrix)

In [ ]:
# Baixando a base de dados direto do repositório da disciplina para o Google Colab
# Fonte original: https://www.kaggle.com/pwnpen/payment

!curl -O https://raw.githubusercontent.com/ahirtonlopes/Machine-Learning-Foundations-and-Classic-Models/main/Datasets/payment_fraud.csv

In [ ]:
# ALTERNATIVA: se preferir subir o arquivo na mão, descomente as linhas abaixo,
# execute esta célula e selecione o payment_fraud.csv do seu computador.

# from google.colab import files
#
# uploaded = files.upload()
#
# for fn in uploaded.keys():
#     print('Arquivo "{name}" com {length} bytes'.format(
#         name=fn, length=len(uploaded[fn])))

In [3]:
# Lendo dados a partir de nosso arquivo .csv
df = pd.read_csv('payment_fraud.csv')

In [4]:
# Amostragem de cabeçalho e cinco primeiras linhas de nosso dataset
df.sample(5)

,accountAgeDays,numItems,localTime,paymentMethod,paymentMethodAgeDays,label
21649,17,1,4.742303,creditcard,0.000694,0
16727,759,1,4.876771,creditcard,606.978472,0
9944,1064,1,4.921349,creditcard,7.868750,0
7025,886,1,4.921318,creditcard,0.000000,0
32304,684,1,4.748314,paypal,349.727778,0


In [5]:
# Verificando linhas e partir do índice de nosso dataset
len(df.index)

39221

In [6]:
# Verificando colunas de nosso dataset
df.columns

Index(['accountAgeDays', 'numItems', 'localTime', 'paymentMethod',
       'paymentMethodAgeDays', 'label'],
      dtype='str')

In [7]:
# Verificando itens únicos de nossa coluna 'paymentMethod' de nosso dataset
(df['paymentMethod'].unique())

<StringArray>
['paypal', 'storecredit', 'creditcard']
Length: 3, dtype: str

In [8]:
# Agrupando número de itens de nossa coluna 'numItems' de nosso dataset
df.groupby('numItems').size().reset_index()

,numItems,0
0,1,37398
1,2,1348
2,3,164
3,4,42
4,5,168
5,6,15
6,7,5
7,8,5
8,9,1
9,10,71


In [9]:
# Tratando nossos dados da coluna 'pamentMethod' para formato numérico - Ref. https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html
df_one_hot = pd.get_dummies(df, columns=['paymentMethod'])

In [10]:
df_one_hot.sample(3)

,accountAgeDays,numItems,localTime,paymentMethodAgeDays,label,paymentMethod_creditcard,paymentMethod_paypal,paymentMethod_storecredit
21082,625,1,4.886641,0.000000,0,False,False,True
13212,2000,1,4.921349,0.235417,0,False,True,False
5777,172,1,4.748314,0.002083,0,True,False,False


In [11]:
# Divisão em sets de treinamento/teste (Regra de Pareto - 80/20 - Ref. https://pt.wikipedia.org/wiki/Princ%C3%ADpio_de_Pareto)
X_train, X_test, y_train, y_test = train_test_split(df_one_hot.drop('label', axis=1), df_one_hot['label'], test_size=0.2, random_state=42)

In [12]:
len(X_train.columns)

7

## Avaliação do Modelo: Compreendendo as Métricas de Desempenho

Após treinar nossos modelos de Machine Learning (Regressão Logística e Árvore de Decisão) para classificar transações como fraude ou não fraude, é crucial avaliar seu desempenho. As métricas de avaliação nos ajudam a entender o quão bem o modelo está realizando suas previsões e quais tipos de erros ele está cometendo.

Para problemas de classificação como o nosso (detecção de fraude), métricas como **Acurácia**, **Precisão** e **Recall** são fundamentais. Além disso, a **Matriz de Confusão** oferece uma visão detalhada dos acertos e erros do modelo.

### Entendendo as Métricas Chave:

*   **Acurácia (Accuracy)**: É a proporção de previsões corretas em relação ao total de previsões. Indica a eficácia geral do modelo.
    *   `Acurácia = (Verdadeiros Positivos + Verdadeiros Negativos) / (Total de Amostras)`

*   **Precisão (Precision)**: Mede a proporção de verdadeiros positivos entre todas as previsões positivas (ou seja, quando o modelo previu 'fraude', quantas vezes estava realmente correto).
    *   É importante quando o custo de um falso positivo é alto (ex: classificar uma transação legítima como fraude, causando inconveniência ao cliente).
    *   `Precisão = Verdadeiros Positivos / (Verdadeiros Positivos + Falsos Positivos)`

*   **Recall (Sensibilidade ou Cobertura)**: Mede a proporção de verdadeiros positivos entre todas as amostras que realmente eram positivas (ou seja, de todas as fraudes reais, quantas o modelo conseguiu detectar).
    *   É importante quando o custo de um falso negativo é alto (ex: não detectar uma fraude real, resultando em perda financeira).
    *   `Recall = Verdadeiros Positivos / (Verdadeiros Positivos + Falsos Negativos)`

*   **Matriz de Confusão (Confusion Matrix)**: Uma tabela que descreve o desempenho de um modelo de classificação em um conjunto de dados de teste para o qual os valores verdadeiros são conhecidos. Ela mostra o número de:
    *   **Verdadeiros Positivos (VP)**: Fraudes corretamente identificadas.
    *   **Verdadeiros Negativos (VN)**: Não-fraudes corretamente identificadas.
    *   **Falsos Positivos (FP)**: Não-fraudes incorretamente identificadas como fraude (Erro Tipo I).
    *   **Falsos Negativos (FN)**: Fraudes incorretamente identificadas como não-fraude (Erro Tipo II).

In [13]:
# Construção de nosso modelo usando Regressão Logística (Ref. https://edisciplinas.usp.br/pluginfile.php/3769787/mod_resource/content/1/09_RegressaoLogistica.pdf)
clf = LogisticRegression(max_iter=100).fit(X_train, y_train)

# Predição em dados de teste
y_pred = clf.predict(X_test)

In [14]:
# Avaliação de Acurácia, Precisão e Recall de nosso modelo
# Atenção à ordem dos argumentos: primeiro o valor real, depois o previsto.
# Se inverter, a precisão e o recall trocam de lugar silenciosamente.
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

acuracia 1.0
precisao 1.0
recall 1.0


In [15]:
# Utilizando Árvore de Decisão
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# Avaliação de Acurácia, Precisão e Recall de nosso modelo
# Atenção à ordem dos argumentos: primeiro o valor real, depois o previsto.
# Se inverter, a precisão e o recall trocam de lugar silenciosamente.
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

acuracia 1.0
precisao 1.0
recall 1.0


---

## Ato 1: cem por cento em tudo? Isso é suspeito

Acurácia 1.0, precisão 1.0 e recall 1.0, nos **dois** modelos. Antes de comemorar, uma regra de ouro: em problema real, métrica perfeita quase nunca significa modelo excelente. Significa **bug**.

Detecção de fraude é um problema difícil por natureza: fraudador imita comportamento legítimo. Nenhum modelo deveria acertar 100% das transações. Então vamos investigar de onde vem esse resultado bom demais.

In [16]:
from sklearn.metrics import confusion_matrix, f1_score

# Primeiro, a matriz de confusão: onde exatamente o modelo acertou e errou?
vn, fp, fn, vp = confusion_matrix(y_test, y_pred).ravel()
print("Matriz de confusão da Árvore de Decisão")
print(f"  Verdadeiros Negativos (não-fraude acertada) : {vn}")
print(f"  Falsos Positivos      (alarme falso)        : {fp}")
print(f"  Falsos Negativos      (fraude que passou)   : {fn}")
print(f"  Verdadeiros Positivos (fraude pega)         : {vp}")
print()

# Quais variáveis a árvore realmente usou para decidir?
print("Importância de cada variável para a árvore:")
for coluna, importancia in sorted(zip(X_train.columns, clf.feature_importances_),
                                  key=lambda x: -x[1]):
    print(f"  {coluna:<28s} {importancia:.3f}")

Matriz de confusão da Árvore de Decisão
  Verdadeiros Negativos (não-fraude acertada) : 7727
  Falsos Positivos      (alarme falso)        : 0
  Falsos Negativos      (fraude que passou)   : 0
  Verdadeiros Positivos (fraude pega)         : 118

Importância de cada variável para a árvore:
  accountAgeDays               1.000
  numItems                     0.000
  localTime                    0.000
  paymentMethodAgeDays         0.000
  paymentMethod_creditcard     0.000
  paymentMethod_paypal         0.000
  paymentMethod_storecredit    0.000


A árvore deu importância **1.000** para `accountAgeDays` e **0.000** para todas as outras seis variáveis. Ou seja, ela decide tudo olhando uma única coluna. Vamos ver o que tem nela:

In [17]:
# Como a idade da conta se distribui entre fraude e não-fraude?
print("accountAgeDays nas transações FRAUDULENTAS:")
print(f"  mínimo: {df[df.label == 1].accountAgeDays.min()}   máximo: {df[df.label == 1].accountAgeDays.max()}")
print()
print("accountAgeDays nas transações LEGÍTIMAS:")
print(f"  mínimo: {df[df.label == 0].accountAgeDays.min()}   máximo: {df[df.label == 0].accountAgeDays.max()}")
print()

# A regra "conta com 1 dia de idade => fraude" sozinha, sem modelo nenhum:
regra = (df.accountAgeDays == 1)
print(f"A regra 'accountAgeDays == 1 => fraude' acerta {(regra == (df.label == 1)).mean():.4%} da base inteira.")

accountAgeDays nas transações FRAUDULENTAS:
  mínimo: 1   máximo: 1

accountAgeDays nas transações LEGÍTIMAS:
  mínimo: 2   máximo: 2000

A regra 'accountAgeDays == 1 => fraude' acerta 100.0000% da base inteira.


### O diagnóstico: vazamento de alvo (*target leakage*)

Está explicado. **Toda** fraude desta base tem `accountAgeDays == 1` e **nenhuma** transação legítima tem. O rótulo que queremos prever é uma função determinística de uma coluna de entrada.

Isso é *vazamento de alvo*: uma variável que, de alguma forma, já carrega a resposta. Acontece muito na prática, e quase sempre por um destes motivos:

*   a variável só é preenchida **depois** que o desfecho aconteceu (ex.: usar "valor estornado" para prever fraude);
*   a base foi construída ou rotulada usando exatamente essa regra;
*   houve contaminação entre treino e teste no pré-processamento.

O modelo não aprendeu a detectar fraude. Ele aprendeu a copiar um marcador. Colocado em produção, com contas reais de idades variadas, ele iria a zero.

**A lição principal desta demo é essa:** quando o resultado vier perfeito, procure o vazamento antes de acreditar.

---

## Ato 2: removendo a coluna que vaza

Vamos refazer tudo sem o `accountAgeDays` e ver o modelo enfrentar o problema de verdade.

Duas mudanças no `train_test_split`: além de tirar a coluna, usamos `stratify=y`. Como as fraudes são só ~1,4% da base, sem estratificação o sorteio pode deixar uma proporção bem diferente de fraudes no treino e no teste. O `stratify` garante que os dois conjuntos mantenham a mesma proporção do original.

In [18]:
# Uma função para não repetir o mesmo bloco de avaliação três vezes
def avaliar(nome, y_true, y_pred):
    vn, fp, fn, vp = confusion_matrix(y_true, y_pred).ravel()
    print(f"{nome}")
    print(f"  acuracia {accuracy_score(y_true, y_pred):.4f} | "
          f"precisao {precision_score(y_true, y_pred, zero_division=0):.4f} | "
          f"recall {recall_score(y_true, y_pred):.4f} | "
          f"f1 {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  fraudes pegas: {vp} de {vp + fn}   |   alarmes falsos: {fp}")
    print()

# Mesma base, sem a coluna que vazava
X2 = df_one_hot.drop(['label', 'accountAgeDays'], axis=1)
y2 = df_one_hot['label']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)

print(f"Fraudes no treino: {y2_train.mean():.4%}  |  no teste: {y2_test.mean():.4%}")
print()

lr2 = LogisticRegression(max_iter=1000).fit(X2_train, y2_train)
dt2 = DecisionTreeClassifier(random_state=42, max_depth=5).fit(X2_train, y2_train)

avaliar("Regressão Logística", y2_test, lr2.predict(X2_test))
avaliar("Árvore de Decisão",   y2_test, dt2.predict(X2_test))

Fraudes no treino: 1.4278%  |  no teste: 1.4277%

Regressão Logística
  acuracia 0.9853 | precisao 0.0000 | recall 0.0000 | f1 0.0000
  fraudes pegas: 0 de 112   |   alarmes falsos: 3

Árvore de Decisão
  acuracia 0.9857 | precisao 0.0000 | recall 0.0000 | f1 0.0000
  fraudes pegas: 0 de 112   |   alarmes falsos: 0



### Leia com atenção o que acabou de acontecer

A acurácia continua ótima, **98,5%**. E o recall é **zero**.

Os dois modelos aprenderam a mesma estratégia: responder "não é fraude" para absolutamente tudo. Como 98,6% das transações realmente não são fraude, essa estratégia burra entrega 98,5% de acurácia. E **nenhuma das 112 fraudes do conjunto de teste foi detectada**.

Se olhássemos só a acurácia, colocaríamos esse modelo em produção achando que ele funciona. Foi a **matriz de confusão** que denunciou: verdadeiros positivos igual a zero.

É por isso que acurácia não serve sozinha em base desbalanceada, e é por isso que precisão, recall e F1 existem.

---

## Ato 3: dizendo ao modelo que fraude custa mais caro

O modelo do Ato 2 não está errado do ponto de vista matemático: ignorar a classe rara é mesmo o que minimiza o erro total. O problema é que, para o negócio, deixar passar uma fraude custa muito mais do que incomodar um cliente legítimo, e o modelo não sabia disso.

O `class_weight='balanced'` corrige isso: ele faz cada erro na classe rara pesar proporcionalmente mais durante o treinamento.

In [19]:
lr3 = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X2_train, y2_train)
dt3 = DecisionTreeClassifier(random_state=42, max_depth=5,
                             class_weight='balanced').fit(X2_train, y2_train)

avaliar("Regressão Logística (balanceada)", y2_test, lr3.predict(X2_test))
avaliar("Árvore de Decisão (balanceada)",   y2_test, dt3.predict(X2_test))

Regressão Logística (balanceada)
  acuracia 0.4894 | precisao 0.0265 | recall 0.9732 | f1 0.0516
  fraudes pegas: 109 de 112   |   alarmes falsos: 4003

Árvore de Decisão (balanceada)
  acuracia 0.7026 | precisao 0.0352 | recall 0.7500 | f1 0.0672
  fraudes pegas: 84 de 112   |   alarmes falsos: 2305



### O trade-off entre precisão e recall, na prática

A regressão logística balanceada passou a pegar **109 das 112 fraudes** (recall 0,97). Em troca, disparou **4.003 alarmes falsos**, e por isso a precisão caiu para 0,027: de cada 100 transações que ela aponta como fraude, menos de 3 são fraude de verdade. A acurácia despencou para 0,49, pior que o modelo inútil do Ato 2.

Compare com os três cenários que vimos:

| Modelo | Acurácia | Precisão | Recall | Fraudes pegas | Alarmes falsos |
|---|---|---|---|---|---|
| Ato 1 (com vazamento) | 1.000 | 1.000 | 1.000 | todas | 0 |
| Ato 2 (sem vazamento) | 0.985 | 0.000 | 0.000 | nenhuma | 0 |
| Ato 3 (balanceado) | 0.489 | 0.027 | 0.973 | quase todas | 4.003 |

**Qual desses é o melhor modelo?** A pergunta não tem resposta técnica, tem resposta de negócio:

*   Se cada fraude não detectada custa um estorno de milhares de reais e um alarme falso custa só uma verificação extra, o Ato 3 é claramente melhor.
*   Se cada alarme falso significa bloquear a compra de um cliente legítimo e perdê-lo, 4.003 bloqueios são inaceitáveis.

O caminho no mundo real fica entre os dois: ajustar o limiar de decisão em vez de usar o 0,5 padrão, buscar variáveis com poder preditivo de verdade (as que sobraram aqui são fracas) e escolher a métrica a otimizar a partir do custo de cada tipo de erro. É por isso que a matéria insiste que **escolher a métrica é uma decisão de projeto, não um detalhe técnico**.

---

### Testando o modelo final com novas transações

Vamos usar o modelo do Ato 3, a **regressão logística balanceada**, para classificar transações novas. Note que os exemplos não têm mais a coluna `accountAgeDays`: ela foi removida justamente por vazar a resposta, e o modelo final nunca a viu.

Além da classificação, vamos imprimir a **probabilidade** que o modelo atribui a cada transação. É esse número que, na prática, você compara com um limiar para decidir. O padrão é 0,5, mas subir ou descer esse corte é exatamente como se navega o trade-off entre precisão e recall que vimos acima.

In [20]:
# As colunas precisam ser as mesmas, e na mesma ordem, do X2_train
novos_exemplos = pd.DataFrame([
    {
        'numItems': 1,
        'localTime': 4.5,
        'paymentMethodAgeDays': 0.0,
        'paymentMethod_creditcard': True,
        'paymentMethod_paypal': False,
        'paymentMethod_storecredit': False
    },  # Exemplo 1: cartão cadastrado hoje mesmo
    {
        'numItems': 2,
        'localTime': 4.9,
        'paymentMethodAgeDays': 1000.0,
        'paymentMethod_creditcard': False,
        'paymentMethod_paypal': True,
        'paymentMethod_storecredit': False
    },  # Exemplo 2: método de pagamento antigo, parece legítimo
    {
        'numItems': 1,
        'localTime': 4.0,
        'paymentMethodAgeDays': 0.1,
        'paymentMethod_creditcard': False,
        'paymentMethod_paypal': False,
        'paymentMethod_storecredit': True
    },  # Exemplo 3: store credit recém-criado
    {
        'numItems': 1,
        'localTime': 4.7,
        'paymentMethodAgeDays': 250.0,
        'paymentMethod_creditcard': True,
        'paymentMethod_paypal': False,
        'paymentMethod_storecredit': False
    },  # Exemplo 4: cliente com histórico
])[X2_train.columns]

print("Novos exemplos para classificação:")
display(novos_exemplos)

predicoes = lr3.predict(novos_exemplos)
probabilidades = lr3.predict_proba(novos_exemplos)[:, 1]

print("\nPrevisões do modelo (0 = Não Fraude, 1 = Fraude):")
for i, (predicao, prob) in enumerate(zip(predicoes, probabilidades), start=1):
    print(f"Exemplo {i}: {predicao}   (probabilidade de fraude: {prob:.1%})")

Novos exemplos para classificação:


,numItems,localTime,paymentMethodAgeDays,paymentMethod_creditcard,paymentMethod_paypal,paymentMethod_storecredit
0,1,4.5,0.0,True,False,False
1,2,4.9,1000.0,False,True,False
2,1,4.0,0.1,False,False,True
3,1,4.7,250.0,True,False,False



Previsões do modelo (0 = Não Fraude, 1 = Fraude):
Exemplo 1: 1   (probabilidade de fraude: 66.1%)
Exemplo 2: 0   (probabilidade de fraude: 0.0%)
Exemplo 3: 1   (probabilidade de fraude: 71.6%)
Exemplo 4: 0   (probabilidade de fraude: 0.0%)
